**Project Goal:** Construct a Machine Learning dataset to predict the success of Liquidity Bootstrapping Pools (LBPs) based on their initial configuration.

**Pipeline Overview:**

1. **Ingestion:** Fetches raw LBP configuration data directly from **Dune Analytics** (Ethereum, Arbitrum, Polygon, Optimism, Base, Avalanche).
2. **Cleaning:**
* Parses raw blockchain arrays and timestamps.
* **Deduplication:** Filters out test pools (<6h) and identifies the unique "True Launch" event for each pool.


3. **Feature Engineering:**
* **Weight Analysis:** Splits array weights into distinct `Project` vs `Reserve` features and calculates the `weight_slope` (velocity of price pressure).
* **Contextual Features:** Derives `swap_fee_pct`, identifies Stablecoin collaterals, and calculates Weekend timing metrics.


4. **Financial Labelling:** Merges configuration data with trading performance metrics (Volume, Retention, Volatility) to generate the final `final_success_score` target.

**Output:** A clean, enriched dataset (`training_dataset.csv`) comprising **961 unique LBPs** ready for model training.

In [1]:
import os
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta, timezone
from dune_client.client import DuneClient
from dune_client.query import QueryBase
from dune_client.types import QueryParameter
from web3 import Web3

In [2]:
# Import DUNE_API_KEY from .env
from dotenv import load_dotenv
load_dotenv()
DUNE_API_KEY = os.getenv("DUNE_API_KEY")
print("DUNE_API_KEY loaded:", DUNE_API_KEY is not None)

DUNE_API_KEY loaded: True


# Table A

In [7]:
# ==============================================================================
# 1. SETUP & CONSTANTS
# ==============================================================================
QUERY_ID = 6570778 # Ensure this ID matches your updated SQL with swapFeePercentage

# TIME WINDOW
START_DATE = datetime(2020, 1, 1) 
END_DATE = datetime.now()
WINDOW_DAYS = 90

# COLLATERAL DEFINITIONS (For identifying Reserve vs Project Token)
# 1. Stablecoins (USDC, DAI, USDT on Eth, Poly, Arb)
STABLE_COLLATERALS = [
    '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48', # USDC (Eth)
    '0x6b175474e89094c44da98b954eedeac495271d0f', # DAI (Eth)
    '0xdac17f958d2ee523a2206206994597c13d831ec7', # USDT (Eth)
    '0x2791bca1f2de4661ed88a30c99a7a9449aa84174', # USDC (Poly)
    '0xc2132d05d31c914a87c6611c10748aeb04b58e8f', # USDT (Poly)
    '0x8f3cf7ad23cd3cadbd9735aff958023239c6a063', # DAI (Poly)
    '0xff970a61a04b1ca14834a43f5de4533ebddb5cc8', # USDC (Arb)
    '0xda10009cbd5d07dd0cecc66161fc93d7c9000da1', # DAI (Arb)
    '0xfd086bc7cd5c481dcc9c85ebe478a1c0b69fcbb9', # USDT (Arb)
]

# 2. All Collaterals (Stables + Volatile like WETH)
ALL_COLLATERALS = STABLE_COLLATERALS + [
    '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2', # WETH (Eth)
    '0x7ceb23fd6bc0add59e62ac25578270cff1b9f619', # WETH (Poly)
    '0x82af49447d8a07e3bd95bd0d56f35241523fbab1', # WETH (Arb)
]

# ==============================================================================
# 2. EXTRACTION ENGINE (Dune Only)
# ==============================================================================

def fetch_data_from_dune():
    dune = DuneClient(DUNE_API_KEY)
    all_rows = []
    
    current_start = START_DATE
    print(f"🚀 Starting Dune Direct Extraction ({START_DATE.date()} -> {END_DATE.date()})...")

    while current_start < END_DATE:
        current_end = current_start + timedelta(days=WINDOW_DAYS)
        
        # Format for Dune {{param}}
        p_start = current_start.strftime("%Y-%m-%d 00:00:00")
        p_end = current_end.strftime("%Y-%m-%d 00:00:00")
        
        print(f"   Extracting window: {p_start} -> {p_end}")
        
        query = QueryBase(
            query_id=QUERY_ID,
            params=[
                QueryParameter.text_type("start_time", p_start),
                QueryParameter.text_type("end_time", p_end)
            ]
        )
        
        try:
            results = dune.run_query(query)
            rows = results.result.rows
            if rows:
                all_rows.extend(rows)
                print(f"     ✅ Found {len(rows)} LBPs")
            else:
                print("     ⚠️ No data")
                
        except Exception as e:
            print(f"     ❌ Error: {e}")
            
        current_start = current_end
        time.sleep(1) # Rate limit politeness

    return pd.DataFrame(all_rows)

# ==============================================================================
# 3. CLEANING ENGINE (Enhanced)
# ==============================================================================

def clean_and_format(df):
    if df.empty: return df
    print(f"\n🧹 Cleaning {len(df)} rows...")

    # 1. PARSE ARRAYS
    # ---------------------------------------------------------
    def parse_list(x):
        if isinstance(x, list): return x
        if isinstance(x, str):
            clean = x.replace('[', '').replace(']', '').replace('"', '').replace("'", "")
            return [i.strip() for i in clean.split(',')] if clean else []
        return []

    df['token_addresses'] = df['token_addresses'].apply(parse_list)
    df['start_weights'] = df['start_weights'].apply(parse_list)
    df['end_weights'] = df['end_weights'].apply(parse_list)

    # 2. NORMALIZE WEIGHTS (Wei -> Decimal)
    # ---------------------------------------------------------
    def normalize_weights(w_list):
        try: return [float(w) / 10**18 for w in w_list]
        except: return []

    df['start_weights'] = df['start_weights'].apply(normalize_weights)
    df['end_weights'] = df['end_weights'].apply(normalize_weights)

    # 3. HANDLE TIMESTAMPS
    # ---------------------------------------------------------
    df['start_time_unix'] = pd.to_numeric(df['start_time_unix'], errors='coerce')
    df['end_time_unix'] = pd.to_numeric(df['end_time_unix'], errors='coerce')

    def get_time_data(row):
        # A. Perfect Data (Schedule Found)
        if pd.notnull(row['start_time_unix']) and pd.notnull(row['end_time_unix']):
            return (
                datetime.fromtimestamp(row['start_time_unix']), 
                datetime.fromtimestamp(row['end_time_unix'])
            )
        # B. Partial Data -> Approx
        start_ts = pd.to_datetime(row['creation_time']).replace(tzinfo=None)
        end_ts = start_ts + timedelta(days=3) 
        return start_ts, end_ts

    time_data = df.apply(get_time_data, axis=1)
    df['start_timestamp'] = [t[0] for t in time_data]
    df['end_timestamp'] = [t[1] for t in time_data]
    
    # Force UTC for calculations
    df['start_timestamp'] = pd.to_datetime(df['start_timestamp'], utc=True)
    df['end_timestamp'] = pd.to_datetime(df['end_timestamp'], utc=True)

    # 4. CALC DURATION & FILTER
    # ---------------------------------------------------------
    df['duration_hours'] = (df['end_timestamp'] - df['start_timestamp']).dt.total_seconds() / 3600
    df = df[df['token_addresses'].apply(lambda x: len(x) == 2)].copy()

    # =========================================================
    # 5. NEW FEATURES: WEIGHT SPLIT & STABLE CHECK
    # =========================================================
    print("   Splitting weights and identifying reserves...")

    def process_weights_and_stable(row):
        tokens = row['token_addresses']
        s_weights = row['start_weights']
        e_weights = row['end_weights']
        
        # Safety Check
        if len(tokens) != 2 or len(s_weights) < 2:
            return pd.Series([0.0, 0.0, 0.0, 0.0, 0])

        # Default: Index 0 is Reserve (Collateral), Index 1 is Project
        res_idx = 0
        proj_idx = 1
        
        # Identify Real Reserve
        t0 = tokens[0].lower().strip()
        t1 = tokens[1].lower().strip()
        
        if t1 in ALL_COLLATERALS:
            res_idx = 1
            proj_idx = 0
        elif t0 in ALL_COLLATERALS:
            res_idx = 0
            proj_idx = 1
            
        # Extract
        try:
            sw_res = float(s_weights[res_idx])
            sw_proj = float(s_weights[proj_idx])
            
            # Handle empty end_weights (Static Pool logic)
            if not e_weights:
                ew_res = sw_res
                ew_proj = sw_proj
            else:
                ew_res = float(e_weights[res_idx])
                ew_proj = float(e_weights[proj_idx])
                
            # Check if Collateral is Stablecoin
            is_stable = 1 if tokens[res_idx].lower() in STABLE_COLLATERALS else 0
            
            return pd.Series([sw_proj, ew_proj, sw_res, ew_res, is_stable])
            
        except:
            return pd.Series([0.0, 0.0, 0.0, 0.0, 0])

    new_cols = ['start_weight_proj', 'end_weight_proj', 'start_weight_reserve', 'end_weight_reserve', 'collateral_is_stable']
    df[new_cols] = df.apply(process_weights_and_stable, axis=1)

    # =========================================================
    # 6. NEW FEATURE: SWAP FEE PCT
    # =========================================================
    print("   Calculating Swap Fees...")
    def calc_fee(x):
        try:
            val = float(x)
            # Heuristic: Raw Wei (10^16 or 10^18) vs Decimal
            if val > 1e14: return val / 1e18
            return val
        except: return 0.01 # Default 1%

    if 'swap_fee_raw' in df.columns:
        df['swap_fee_pct'] = df['swap_fee_raw'].apply(calc_fee)
    else:
        df['swap_fee_pct'] = 0.01

    # =========================================================
    # 7. NEW FEATURES: WEEKEND MATH
    # =========================================================
    print("   Calculating Weekend Metrics...")
    
    # is_weekend (Launch day is Sat(5) or Sun(6))
    df['is_weekend'] = df['start_timestamp'].dt.weekday.apply(lambda x: 1 if x >= 5 else 0)

    # weekend_pct (Duration overlap)
    def calc_weekend_pct(row):
        start = row['start_timestamp']
        duration = row['duration_hours']
        if duration <= 0: return 0.0
        
        # Fast Approximation for large datasets
        # If duration is massive, we cap loop, otherwise create range
        try:
            # Create hourly timestamps
            hours = pd.date_range(start, periods=int(duration)+1, freq='h')
            weekend_hours = sum(1 for h in hours if h.weekday() >= 5)
            return weekend_hours / len(hours)
        except:
            return 0.0

    df['weekend_pct'] = df.apply(calc_weekend_pct, axis=1)

    # Placeholder for Liquidity (Requested to be blank/zero)
    df['initial_liquidity_usd'] = 0.0

    # 8. FINAL SCHEMA
    final_cols = [
        'pool_address', 'chain', 'version', 'token_addresses',
        'start_timestamp', 'end_timestamp', 'duration_hours',
        'start_weight_proj', 'end_weight_proj', 
        'start_weight_reserve', 'end_weight_reserve',
        'swap_fee_pct', 'collateral_is_stable', 
        'is_weekend', 'weekend_pct',
        'initial_liquidity_usd'
    ]
    
    # Ensure all columns exist before filtering
    available_cols = [c for c in final_cols if c in df.columns]
    
    return df[available_cols]

if __name__ == "__main__":
    df_raw = fetch_data_from_dune()
    if not df_raw.empty:
        df_final = clean_and_format(df_raw)
        df_final.to_csv("table_a_complete.csv", index=False)
        print(f"\n🎉 Success! Saved {len(df_final)} LBPs to table_a_complete.csv")
        print(df_final.info())

🚀 Starting Dune Direct Extraction (2020-01-01 -> 2026-01-22)...
   Extracting window: 2020-01-01 00:00:00 -> 2020-03-31 00:00:00
     ⚠️ No data
   Extracting window: 2020-03-31 00:00:00 -> 2020-06-29 00:00:00
     ⚠️ No data
   Extracting window: 2020-06-29 00:00:00 -> 2020-09-27 00:00:00
     ⚠️ No data
   Extracting window: 2020-09-27 00:00:00 -> 2020-12-26 00:00:00
     ⚠️ No data
   Extracting window: 2020-12-26 00:00:00 -> 2021-03-26 00:00:00
     ⚠️ No data
   Extracting window: 2021-03-26 00:00:00 -> 2021-06-24 00:00:00
     ⚠️ No data
   Extracting window: 2021-06-24 00:00:00 -> 2021-09-22 00:00:00
     ✅ Found 22 LBPs
   Extracting window: 2021-09-22 00:00:00 -> 2021-12-21 00:00:00
     ✅ Found 682 LBPs
   Extracting window: 2021-12-21 00:00:00 -> 2022-03-21 00:00:00
     ✅ Found 351 LBPs
   Extracting window: 2022-03-21 00:00:00 -> 2022-06-19 00:00:00
     ✅ Found 383 LBPs
   Extracting window: 2022-06-19 00:00:00 -> 2022-09-17 00:00:00
     ✅ Found 349 LBPs
   Extracting wi

In [9]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================
FILE_INPUT = "table_a_complete.csv" 
OUTPUT_FILE = "table_a_final_enriched.csv"

# FILTERS (Matches old table_a_final logic)
MIN_DURATION_HOURS = 6   
MAX_DURATION_HOURS = 1440 

# Collaterals
STABLE_COLLATERALS = [
    '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48', '0x6b175474e89094c44da98b954eedeac495271d0f',
    '0xdac17f958d2ee523a2206206994597c13d831ec7', '0x2791bca1f2de4661ed88a30c99a7a9449aa84174',
    '0xc2132d05d31c914a87c6611c10748aeb04b58e8f', '0x8f3cf7ad23cd3cadbd9735aff958023239c6a063',
    '0xff970a61a04b1ca14834a43f5de4533ebddb5cc8', '0xda10009cbd5d07dd0cecc66161fc93d7c9000da1',
    '0xfd086bc7cd5c481dcc9c85ebe478a1c0b69fcbb9'
]
WETH_COLLATERALS = [
    '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2', '0x7ceb23fd6bc0add59e62ac25578270cff1b9f619',
    '0x82af49447d8a07e3bd95bd0d56f35241523fbab1'
]
ALL_COLLATERALS = STABLE_COLLATERALS + WETH_COLLATERALS

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
def main():
    print("🚀 Starting Final Cleanup & Enrichment...")

    try:
        df = pd.read_csv(FILE_INPUT)
    except FileNotFoundError:
        print("❌ Input file not found.")
        return

    # 1. CLEAN TYPES
    df['start_timestamp'] = pd.to_datetime(df['start_timestamp'], utc=True)
    df['end_timestamp'] = pd.to_datetime(df['end_timestamp'], utc=True)
    df['duration_hours'] = (df['end_timestamp'] - df['start_timestamp']).dt.total_seconds() / 3600

    # 2. FILTER FIRST (Crucial for Data Quality)
    valid_duration = (df['duration_hours'] >= MIN_DURATION_HOURS) & (df['duration_hours'] <= MAX_DURATION_HOURS)
    df = df[valid_duration].copy()

    # 3. DEDUPLICATE SECOND
    df = df.sort_values(by=['pool_address', 'start_timestamp'], ascending=[True, True])
    df = df.drop_duplicates(subset=['pool_address'], keep='first')

    # 4. ENRICHMENT
    print("   Enriching (Liquidity & Slope)...")

    # B. Weight Slope (Re-calculate to ensure accuracy)
    # Formula: Abs(Start - End) / Duration
    def calc_slope(row):
        try:
            change = abs(row['end_weight_proj'] - row['start_weight_proj'])
            return change / row['duration_hours'] if row['duration_hours'] > 0 else 0.0
        except: return 0.0
        
    df['weight_slope'] = df.apply(calc_slope, axis=1)

    # 5. SAVE
    output_cols = [
        'pool_address', 'chain', 'version', 
        'start_timestamp', 'duration_hours',
        'start_weight_proj', 'end_weight_proj', 
        'start_weight_reserve', 'end_weight_reserve',
        'weight_slope',
        'swap_fee_pct', 'collateral_is_stable', 
        'is_weekend', 'weekend_pct',
    ]
    
    final_cols = [c for c in output_cols if c in df.columns]
    
    df[final_cols].to_csv(OUTPUT_FILE, index=False)
    print(f"🎉 Success! Saved {OUTPUT_FILE} ({len(df)} rows)")
    print(df[final_cols].head())

if __name__ == "__main__":
    main()

🚀 Starting Final Cleanup & Enrichment...
   Enriching (Liquidity & Slope)...
🎉 Success! Saved table_a_final_enriched.csv (961 rows)
                                    pool_address     chain  version  \
1209  0x0017c363b29d8f86d82e9681552685f68f34b7e4  ethereum        2   
1523  0x0022b6e4ff3ddbf0c36c7c6c7c7f3062f36be5f8  ethereum        2   
1206  0x006962f9de1aa0639f893aff1f08eddf40a67ad5  ethereum        2   
2410  0x011e612c57ea6932755bd5ec839d291ecf7ca691  arbitrum        2   
2204  0x0129dff11b02c4d542aa0bdc7e99f65183212c32  ethereum        2   

               start_timestamp  duration_hours  start_weight_proj  \
1209 2022-05-24 20:30:01+00:00       72.000000               0.99   
1523 2022-08-18 18:00:00+00:00       72.000000               0.99   
1206 2022-05-27 04:00:01+00:00       72.000000               0.99   
2410 2024-01-08 12:30:01+00:00      168.499722               0.80   
2204 2023-09-04 09:00:00+00:00      144.000000               0.99   

      end_weight_proj  sta

# Table B

In [10]:
# SCORING WEIGHTS
W_RETENTION = 0.40
W_UNIQUE    = 0.20
W_DUMP      = 0.20
W_VOL       = 0.20

In [5]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================
INPUT_FILE = "table_a_final.csv"
OUTPUT_FILE = "table_b_advanced.csv"

# COLLATERALS: Used to identify which token is the "Project Token"
# If a pair is [ProjectToken, USDC], we know USDC is the collateral.
COLLATERALS = [
    '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2', # WETH (Eth)
    '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48', # USDC (Eth)
    '0x6b175474e89094c44da98b954eedeac495271d0f', # DAI (Eth)
    '0xdac17f958d2ee523a2206206994597c13d831ec7', # USDT (Eth)
    '0x2791bca1f2de4661ed88a30c99a7a9449aa84174', # USDC (Poly)
    '0x7ceb23fd6bc0add59e62ac25578270cff1b9f619', # WETH (Poly)
    '0xc2132d05d31c914a87c6611c10748aeb04b58e8f', # USDT (Poly)
    '0x8f3cf7ad23cd3cadbd9735aff958023239c6a063', # DAI (Poly)
    '0x82af49447d8a07e3bd95bd0d56f35241523fbab1', # WETH (Arb)
    '0xff970a61a04b1ca14834a43f5de4533ebddb5cc8', # USDC (Arb)
    '0xda10009cbd5d07dd0cecc66161fc93d7c9000da1', # DAI (Arb)
    '0xfd086bc7cd5c481dcc9c85ebe478a1c0b69fcbb9', # USDT (Arb)
]

# ==============================================================================
# 1. FETCH RAW TRADES (Updated for Granularity)
# ==============================================================================
def fetch_raw_trades(pool_list):
    dune = DuneClient(DUNE_API_KEY)
    
    # Batching to avoid query size limits
    chunk_size = 50
    chunks = [pool_list[i:i + chunk_size] for i in range(0, len(pool_list), chunk_size)]
    all_trades = []
    
    print(f"🚀 Fetching Detailed Trades for {len(pool_list)} pools...")

    for i, chunk in enumerate(chunks):
        # Prepare addresses for SQL (Binary conversion logic)
        pool_str = ",".join([f"FROM_HEX('{addr.replace('0x', '')}')" for addr in chunk])
        
        # SQL: Fetch everything needed for advanced metrics
        sql = f"""
        SELECT 
            '0x' || to_hex(project_contract_address) as pool_address,
            block_number,
            block_time,
            '0x' || to_hex(tx_from) as trader_a,
            '0x' || to_hex(token_bought_address) as token_bought,
            '0x' || to_hex(token_sold_address) as token_sold,
            token_bought_amount,
            token_sold_amount,
            amount_usd
        FROM dex.trades
        WHERE project_contract_address IN ({pool_str})
        AND amount_usd > 0
        ORDER BY block_number ASC
        """
        
        print(f"   Batch {i+1}/{len(chunks)}: Querying Dune...")
        try:
            results = dune.run_sql(sql)
            data = results.get_rows()
            all_trades.extend(data)
            print(f"     ✅ Found {len(data)} trades")
        except Exception as e:
            print(f"     ❌ Error: {e}")
        
        time.sleep(1)

    return pd.DataFrame(all_trades)

# ==============================================================================
# 2. CALCULATE ADVANCED METRICS
# ==============================================================================
def calculate_advanced_metrics(trades_df, pools_df):
    print("\n🧠 Calculating Advanced Metrics (Volatility, Retention, Bots)...")
    results = []
    
    # Pre-process trades
    trades_df['amount_usd'] = pd.to_numeric(trades_df['amount_usd'], errors='coerce')
    trades_df['token_bought_amount'] = pd.to_numeric(trades_df['token_bought_amount'], errors='coerce')
    trades_df['token_sold_amount'] = pd.to_numeric(trades_df['token_sold_amount'], errors='coerce')
    trades_df['block_number'] = pd.to_numeric(trades_df['block_number'], errors='coerce')

    for pool_addr in pools_df['pool_address'].unique():
        # Default "Ghost Pool" Row (All Zeros)
        ghost_row = {
            'pool_address': pool_addr,
            'volatility_score': 0.0,
            'price_retention': 0.0,
            'unique_buyers': 0,
            'bot_tx_ratio': 0.0,
            'volume_usd': 0.0,
            'dump_pressure': 10.0, # Max penalty
            'final_success_score': 0.0
        }

        # 1. CHECK: TRADES EXIST?
        pt = trades_df[trades_df['pool_address'] == pool_addr].copy()
        
        if pt.empty:
            # Case A: No trades found -> It's a Ghost Pool
            results.append(ghost_row)
            continue

        # 2. CHECK: PROJECT TOKEN IDENTIFIED?
        tokens_in_trades = set(pt['token_bought'].unique()) | set(pt['token_sold'].unique())
        project_token = None
        for t in tokens_in_trades:
            if t not in COLLATERALS:
                project_token = t
                break
        
        if not project_token:
            # Case B: Weird pair (e.g. USDC-DAI) -> Treat as dead/noise
            results.append(ghost_row)
            continue

        # 3. CHECK: VALID PRICES?
        def get_price_and_direction(row):
            if row['token_bought'] == project_token:
                price = row['amount_usd'] / row['token_bought_amount'] if row['token_bought_amount'] else 0
                return price, 'buy'
            else:
                price = row['amount_usd'] / row['token_sold_amount'] if row['token_sold_amount'] else 0
                return price, 'sell'

        pt[['price', 'direction']] = pt.apply(lambda x: pd.Series(get_price_and_direction(x)), axis=1)
        pt = pt[pt['price'] > 0]

        if pt.empty:
            # Case C: Trades existed but had 0 value/price -> Ghost Pool
            results.append(ghost_row)
            continue

        # ==========================================
        # IF WE SURVIVED, CALCULATE REAL METRICS
        # ==========================================
        
        # --- METRIC 1: VOLATILITY ---
        mean_price = pt['price'].mean()
        std_price = pt['price'].std()
        if pd.isna(std_price): volatility_score = 0.0 # Single trade
        else: volatility_score = (std_price / mean_price) if mean_price > 0 else 0

        # --- METRIC 2: RETENTION ---
        start_price = pt.head(5)['price'].mean()
        end_price = pt.tail(5)['price'].mean()
        price_retention = (end_price / start_price) if start_price > 0 else 0

        # --- METRIC 3: UNIQUE BUYERS ---
        unique_buyers = pt[pt['direction'] == 'buy']['trader_a'].nunique()

        # --- METRIC 4: BOT RATIO ---
        start_block = pt['block_number'].min()
        bot_trades = pt[pt['block_number'] <= start_block + 3]
        bot_tx_ratio = len(bot_trades) / len(pt)

        # --- METRIC 5: DUMP PRESSURE ---
        buy_vol = pt[pt['direction'] == 'buy']['amount_usd'].sum()
        sell_vol = pt[pt['direction'] == 'sell']['amount_usd'].sum()
        dump_pressure = (sell_vol / buy_vol) if buy_vol > 0 else 10.0

        # --- METRIC 6: VOLUME ---
        volume_usd = pt['amount_usd'].sum()

        # --- FINAL SCORE ---
        # Normalize
        s_retention = min(price_retention, 5.0) / 5.0
        s_unique = min(unique_buyers, 500) / 500.0
        s_dump = 1.0 / (dump_pressure + 1)
        s_vol = 1.0 - min(volatility_score, 1.0)

        final_success_score = (
            (W_RETENTION * s_retention) +
            (W_UNIQUE    * s_unique) + 
            (W_DUMP      * s_dump) +
            (W_VOL       * s_vol)
        )

        results.append({
            'pool_address': pool_addr,
            'volatility_score': round(volatility_score, 4),
            'price_retention': round(price_retention, 4),
            'unique_buyers': unique_buyers,
            'bot_tx_ratio': round(bot_tx_ratio, 4),
            'volume_usd': round(volume_usd, 2),
            'dump_pressure': round(dump_pressure, 4),
            'final_success_score': round(final_success_score, 4)
        })

    return pd.DataFrame(results)

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # 1. Load Pools
    try:
        df_a = pd.read_csv(INPUT_FILE)
        df_a['pool_address'] = df_a['pool_address'].str.lower()
        pools = df_a['pool_address'].unique().tolist()
    except:
        print("❌ Input file not found.")
        exit()

    # 2. Fetch Detailed Trades
    df_trades = fetch_raw_trades(pools)

    # 3. Calculate Advanced Metrics
    if not df_trades.empty:
        # Normalize addresses
        cols_to_lower = ['pool_address', 'trader_a', 'token_bought', 'token_sold']
        for c in cols_to_lower:
            df_trades[c] = df_trades[c].str.lower()

        df_b = calculate_advanced_metrics(df_trades, df_a)
        
        # 4. Save
        df_b.to_csv(OUTPUT_FILE, index=False)
        print(f"\n🎉 Saved Advanced Table B to {OUTPUT_FILE}")
        print(df_b.head())

🚀 Fetching Detailed Trades for 961 pools...
   Batch 1/20: Querying Dune...
     ✅ Found 20696 trades
   Batch 2/20: Querying Dune...
     ✅ Found 18486 trades
   Batch 3/20: Querying Dune...
     ✅ Found 14165 trades
   Batch 4/20: Querying Dune...
     ✅ Found 21137 trades
   Batch 5/20: Querying Dune...
     ✅ Found 6119 trades
   Batch 6/20: Querying Dune...
     ✅ Found 20656 trades
   Batch 7/20: Querying Dune...
     ✅ Found 7603 trades
   Batch 8/20: Querying Dune...
     ✅ Found 9383 trades
   Batch 9/20: Querying Dune...
     ✅ Found 22371 trades
   Batch 10/20: Querying Dune...
     ✅ Found 8438 trades
   Batch 11/20: Querying Dune...
     ✅ Found 13253 trades
   Batch 12/20: Querying Dune...
     ✅ Found 17098 trades
   Batch 13/20: Querying Dune...
     ✅ Found 16254 trades
   Batch 14/20: Querying Dune...
     ✅ Found 12984 trades
   Batch 15/20: Querying Dune...
     ✅ Found 11537 trades
   Batch 16/20: Querying Dune...
     ✅ Found 25598 trades
   Batch 17/20: Querying 

In [ ]:
# Collateral Map (Normalized)
COLLATERAL_MAP = {
    '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2': {'symbol': 'WETH', 'decimals': 18},
    '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48': {'symbol': 'USDC', 'decimals': 6},
    '0x6b175474e89094c44da98b954eedeac495271d0f': {'symbol': 'DAI',  'decimals': 18},
    '0xdac17f958d2ee523a2206206994597c13d831ec7': {'symbol': 'USDT', 'decimals': 6},
    '0x2791bca1f2de4661ed88a30c99a7a9449aa84174': {'symbol': 'USDC', 'decimals': 6},
    '0x7ceb23fd6bc0add59e62ac25578270cff1b9f619': {'symbol': 'WETH', 'decimals': 18},
    '0xc2132d05d31c914a87c6611c10748aeb04b58e8f': {'symbol': 'USDT', 'decimals': 6},
    '0x8f3cf7ad23cd3cadbd9735aff958023239c6a063': {'symbol': 'DAI',  'decimals': 18},
    '0x82af49447d8a07e3bd95bd0d56f35241523fbab1': {'symbol': 'WETH', 'decimals': 18},
    '0xff970a61a04b1ca14834a43f5de4533ebddb5cc8': {'symbol': 'USDC', 'decimals': 6},
    '0xda10009cbd5d07dd0cecc66161fc93d7c9000da1': {'symbol': 'DAI',  'decimals': 18},
    '0xfd086bc7cd5c481dcc9c85ebe478a1c0b69fcbb9': {'symbol': 'USDT', 'decimals': 6},
}

# ==============================================================================
# 1. FETCH TRADES (With strict Block Number casting)
# ==============================================================================
def fetch_raw_trades(pool_list):
    dune = DuneClient(DUNE_API_KEY)
    print(f"🚀 Fetching Trades for {len(pool_list)} pools...")
    chunk_size = 50
    chunks = [pool_list[i:i + chunk_size] for i in range(0, len(pool_list), chunk_size)]
    all_trades = []

    for i, chunk in enumerate(chunks):
        pool_str = ",".join([f"FROM_HEX('{addr.replace('0x', '')}')" for addr in chunk])
        
        # We explicitly CAST block_number to integer in SQL to be safe
        sql = f"""
        SELECT 
            '0x' || to_hex(project_contract_address) as pool_address,
            CAST(block_number AS DOUBLE) as block_number, 
            block_time,
            '0x' || to_hex(tx_from) as trader_a,
            '0x' || to_hex(token_bought_address) as token_bought,
            '0x' || to_hex(token_sold_address) as token_sold,
            token_bought_amount, token_sold_amount, amount_usd
        FROM dex.trades
        WHERE project_contract_address IN ({pool_str})
        AND amount_usd > 0
        ORDER BY block_number ASC
        """
        try:
            res = dune.run_sql(sql)
            all_trades.extend(res.get_rows())
            print(f"   Batch {i+1}: Found {len(res.get_rows())} trades")
        except Exception as e:
            print(f"   ❌ Error Batch {i+1}: {e}")
        time.sleep(1)

    return pd.DataFrame(all_trades)

# ==============================================================================
# 2. CALCULATE METRICS (With Debug Prints)
# ==============================================================================
def calculate_metrics(trades_df, pools_df):
    print("\n🧠 Calculating Metrics (Debug Mode)...")
    results = []

    # 1. Strict Type Conversion
    trades_df['amount_usd'] = pd.to_numeric(trades_df['amount_usd'], errors='coerce').fillna(0)
    trades_df['block_number'] = pd.to_numeric(trades_df['block_number'], errors='coerce') # Crucial for Bots
    trades_df['token_bought_amount'] = pd.to_numeric(trades_df['token_bought_amount'], errors='coerce')
    trades_df['token_sold_amount'] = pd.to_numeric(trades_df['token_sold_amount'], errors='coerce')
    trades_df['block_time'] = pd.to_datetime(trades_df['block_time'])

    norm_map = {k.lower(): v for k, v in COLLATERAL_MAP.items()}

    debug_counter = 0

    for idx, pool_row in pools_df.iterrows():
        pid = pool_row['pool_address']
        liq_usd = pool_row.get('initial_liquidity_usd', 0)
        
        # Filter trades for this pool
        pt = trades_df[trades_df['pool_address'] == pid].copy()
        pt = pt.sort_values('block_number') # Ensure sorted for "Start/End" logic

        # --- DEBUG: Why is it zero? ---
        if pt.empty:
            if debug_counter < 5: print(f"⚠️ Pool {pid[:8]}... : No trades found in DF match.")
            # Ghost Row
            results.append({
                'pool_address': pid,
                'volume_usd': 0, 'unique_buyers': 0, 'price_retention': 0,
                'volatility_score': 0, 'bot_tx_ratio': 0, 'dump_pressure': 10,
                'volume_time_skew': 0.5, 'whale_dominance_pct': 0,
                'turnover_ratio': 0, 'price_discovery_stability': 0, 'bot_extraction_usd': 0
            })
            debug_counter += 1
            continue

        # Identify Project Token
        tokens = set(pt['token_bought'].unique()) | set(pt['token_sold'].unique())
        project_token = next((t for t in tokens if t not in norm_map), None)
        
        if not project_token:
            if debug_counter < 5: print(f"⚠️ Pool {pid[:8]}... : Could not identify Project Token. Tokens: {tokens}")
            continue 

        # Calculate Prices
        def get_price(row):
            if row['token_bought'] == project_token:
                val = float(row['token_bought_amount'])
                return (row['amount_usd'] / val) if val else 0, 'buy'
            val = float(row['token_sold_amount'])
            return (row['amount_usd'] / val) if val else 0, 'sell'

        pt[['price', 'direction']] = pt.apply(lambda x: pd.Series(get_price(x)), axis=1)
        pt = pt[pt['price'] > 0] # Filter bad prices

        # --- METRIC CALCULATION ---
        
        # 1. BOT RATIO (The problematic one)
        # Logic: Trades in the first 5 blocks relative to start
        if pt['block_number'].isna().all():
            bot_ratio = 0
            bot_pnl = 0
            if debug_counter < 5: print(f"⚠️ Pool {pid[:8]}... : Block Numbers are all NaN")
        else:
            start_blk = pt['block_number'].min()
            # Bots = trades within 5 blocks of start
            bots = pt[pt['block_number'] <= (start_blk + 5)]
            bot_ratio = len(bots) / len(pt)
            
            # Bot PnL
            bot_buy = bots[bots['direction']=='buy']['amount_usd'].sum()
            bot_sell = bots[bots['direction']=='sell']['amount_usd'].sum()
            bot_pnl = bot_sell - bot_buy

        # 2. PRICE DISCOVERY (The other problematic one)
        # Logic: Volatility of the last 10% of trades
        if len(pt) > 10:
            last_10 = pt.tail(int(len(pt)*0.1))
            disc_stab = last_10['price'].std() / last_10['price'].mean()
            if pd.isna(disc_stab): disc_stab = 0
        else:
            disc_stab = 0 # Not enough data

        # 3. Standard Metrics
        volume_usd = pt['amount_usd'].sum()
        unique_buyers = pt['trader_a'].nunique()
        volatility = pt['price'].std() / pt['price'].mean() if len(pt) > 1 else 0
        retention = (pt.tail(5)['price'].mean() / pt.head(5)['price'].mean()) if len(pt) > 5 else 0
        
        buy_vol = pt[pt['direction']=='buy']['amount_usd'].sum()
        sell_vol = pt[pt['direction']=='sell']['amount_usd'].sum()
        dump_pressure = sell_vol / buy_vol if buy_vol > 0 else 10.0

        # 4. Activity Skew
        start_t = pt['block_time'].min()
        duration = (pt['block_time'].max() - start_t).total_seconds()
        if duration > 0:
            pt['time_norm'] = (pt['block_time'] - start_t).dt.total_seconds() / duration
            skew = (pt['time_norm'] * pt['amount_usd']).sum() / volume_usd
        else: skew = 0.5
        
        threshold = pt['amount_usd'].quantile(0.99)
        whale_dom = pt[pt['amount_usd'] >= threshold]['amount_usd'].sum() / volume_usd if volume_usd > 0 else 0
        turnover = volume_usd / liq_usd if liq_usd > 0 else 0

        results.append({
            'pool_address': pid,
            'volume_usd': round(volume_usd, 2),
            'unique_buyers': unique_buyers,
            'price_retention': round(retention, 4),
            'volatility_score': round(volatility, 4),
            'dump_pressure': round(dump_pressure, 4),
            'volume_time_skew': round(skew, 4),
            'whale_dominance_pct': round(whale_dom, 4),
            'turnover_ratio': round(turnover, 2),
            'bot_tx_ratio': round(bot_ratio, 4),
            'bot_extraction_usd': round(bot_pnl, 2),
            'price_discovery_stability': round(disc_stab, 4)
        })

    return pd.DataFrame(results)

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    df_a = pd.read_csv(INPUT_FILE)
    df_a['pool_address'] = df_a['pool_address'].str.lower()
    
    # We skip liquidity fetch here to focus on fixing the Zero-Columns first
    # (Assuming you can run the Vault fetch separately or reuse the previous logic)
    df_a['initial_liquidity_usd'] = 100000 # Placeholder to prevent Div/0 errors during debug

    # 2. Fetch Trades
    df_trades = fetch_raw_trades(df_a['pool_address'].unique().tolist())
    
    # 3. Calculate
    if not df_trades.empty:
        # Normalize everything
        for c in ['pool_address', 'trader_a', 'token_bought', 'token_sold']:
            df_trades[c] = df_trades[c].str.lower()
        
        df_b = calculate_metrics(df_trades, df_a)
        df_b.to_csv('improved_'+OUTPUT_FILE, index=False)
        print(f"\n🎉 Saved {'improved_'+OUTPUT_FILE}")
        print(df_b[['pool_address', 'bot_tx_ratio', 'bot_extraction_usd', 'price_discovery_stability']].head(10))

🚀 Fetching Trades for 961 pools...
   Batch 1: Found 20716 trades
   Batch 2: Found 18486 trades
   Batch 3: Found 14167 trades
   Batch 4: Found 21137 trades
   Batch 5: Found 6119 trades
   Batch 6: Found 20656 trades
   Batch 7: Found 7603 trades
   Batch 8: Found 9383 trades
   Batch 9: Found 22372 trades
   Batch 10: Found 8439 trades
   Batch 11: Found 13253 trades
   Batch 12: Found 17098 trades
   Batch 13: Found 16254 trades
   Batch 14: Found 12984 trades
   Batch 15: Found 11549 trades
   Batch 16: Found 25601 trades
   Batch 17: Found 10509 trades
   Batch 18: Found 6435 trades
   Batch 19: Found 15490 trades
   Batch 20: Found 2192 trades

🧠 Calculating Metrics (Debug Mode)...
⚠️ Pool 0x0129df... : No trades found in DF match.
⚠️ Pool 0x02be32... : No trades found in DF match.
⚠️ Pool 0x040722... : No trades found in DF match.
⚠️ Pool 0x049895... : No trades found in DF match.
⚠️ Pool 0x04d0b5... : No trades found in DF match.

🎉 Saved fixed_table_b_advanced.csv
          

# Creating training dataset

In [ ]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================
# Use the CLEANED files you generated in previous steps
FILE_A_CLEAN = "table_a_final_enriched.csv"
FILE_B_FIXED = "table_b_advanced.csv"
OUTPUT_TRAIN = "training_dataset.csv"

def main():
    print("🚀 Starting Simple Merge...")

    # 1. LOAD
    try:
        df_a = pd.read_csv(FILE_A_CLEAN)
        df_b = pd.read_csv(FILE_B_FIXED)
    except FileNotFoundError:
        print(f"❌ Error: Could not find {FILE_A_CLEAN} or {FILE_B_FIXED}.")
        print("   Make sure you run 'clean_lbp_data.py' and 'fix_table_b.py' first.")
        return

    print(f"   Loaded Table A: {len(df_a)} rows")
    print(f"   Loaded Table B: {len(df_b)} rows")

    # 2. STANDARDIZE KEYS
    df_a['pool_address'] = df_a['pool_address'].str.lower().str.strip()
    df_b['pool_address'] = df_b['pool_address'].str.lower().str.strip()

    # 3. MERGE
    # We only need the Target Variable from Table B
    # Inner Join ensures we only train on pools that have BOTH config and financial data
    print("   Merging...")
    df_train = pd.merge(
        df_a, 
        df_b[['pool_address', 'final_success_score']], 
        on='pool_address', 
        how='inner'
    )

    # 4. SELECT FEATURES FOR ML
    # We select only the columns valid for t=0 prediction
    TRAINING_COLS = [
        # 'pool_address',
        'chain',
        'duration_hours',
        'start_weight_proj',
        'end_weight_proj',
        'start_weight_reserve',
        'end_weight_reserve',
        'weight_slope',
        'swap_fee_pct',
        'collateral_is_stable',
        'is_weekend',
        'weekend_pct',
        'final_success_score'
    ]
    
    # Filter columns if they exist
    cols_to_save = [c for c in TRAINING_COLS if c in df_train.columns]
    df_train = df_train[cols_to_save]

    # 5. SAVE
    df_train.to_csv(OUTPUT_TRAIN, index=False)
    print(f"🎉 Saved {OUTPUT_TRAIN} with {len(df_train)} rows.")
    print(df_train.head())

if __name__ == "__main__":
    main()

🚀 Starting Simple Merge...
   Loaded Table A: 961 rows
   Loaded Table B: 961 rows
   Merging...
🎉 Saved training_dataset.csv with 961 rows.
      chain  duration_hours  start_weight_proj  end_weight_proj  \
0  ethereum       72.000000               0.99             0.01   
1  ethereum       72.000000               0.99             0.01   
2  ethereum       72.000000               0.99             0.01   
3  arbitrum      168.499722               0.80             0.20   
4  ethereum      144.000000               0.99             0.01   

   start_weight_reserve  end_weight_reserve  weight_slope  swap_fee_pct  \
0                  0.01                0.99      0.013611         0.025   
1                  0.01                0.99      0.013611         0.030   
2                  0.01                0.99      0.013611         0.025   
3                  0.20                0.80      0.003561         0.025   
4                  0.01                0.99      0.006806         0.010   

   co